# Additional MSWEGNN metrics

This notebook post-processes the `.npz` files produced by `train.py --with_test True`. It calculates depth MAE, discharge MAE, optional unit-discharge MAE, CSI at 0.05 m and 0.30 m, and inference-time mean/standard deviation across events.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import shapefile

def shp_values(filepath, column):
    reader = shapefile.Reader(str(filepath))
    field_names = [
        field[0]
        for field in reader.fields
        if field[0] != "DeletionFlag"
    ]

    column_index = field_names.index(column)
    return np.array([
        record[column_index]
        for record in reader.iterRecords()
    ])

In [16]:
# Adjust these paths if your results are stored elsewhere.
PROJECT_ROOT = Path.cwd()
# METRICS_DIR = PROJECT_ROOT / 'saved_metrics'
METRICS_DIR = PROJECT_ROOT / 'saved_metrics'
RAW_DIR = PROJECT_ROOT / 'data_mswegnn' / 'datasets' / 'raw'
TEST_CSV = RAW_DIR / 'test.csv'
OUTPUT_CSV = PROJECT_ROOT / 'additional_mswegnn_metrics.csv'

# Use one model run, or set to None to process every matching file.
MODEL_PREFIX = "DUALFloodGNN_2026-09-27_08-17-01"  # e.g. 'DUALFloodGNN_2026-09-27_12-30-00'
UNIT_DISCHARGE = True


In [14]:
summary = pd.read_csv(TEST_CSV)
summary['Run_ID'] = summary['Run_ID'].astype(str)
summary_by_run = summary.set_index('Run_ID')

def relative_raw_path(value):
    value = str(value).replace("\\", "/")
    return RAW_DIR / Path(value)

def csi(pred_depth, target_depth, threshold):
    pred_flooded = pred_depth > threshold
    target_flooded = target_depth > threshold
    tp = np.logical_and(pred_flooded, target_flooded).sum()
    fp = np.logical_and(pred_flooded, ~target_flooded).sum()
    fn = np.logical_and(~pred_flooded, target_flooded).sum()
    denominator = tp + fp + fn
    return float(tp / denominator) if denominator else np.nan

def mean_sd(values):
    values = np.asarray(values, dtype=float)
    return float(np.mean(values)), float(np.std(values, ddof=1)) if len(values) > 1 else 0.0


In [15]:
files = sorted(METRICS_DIR.glob('*_runid_*_test_metrics.npz'))
if MODEL_PREFIX is not None:
    files = [path for path in files if path.name.startswith(MODEL_PREFIX + '_runid_')]
if not files:
    raise FileNotFoundError(f'No metric files found in {METRICS_DIR}')

rows = []
for path in files:
    match = re.search(r'_runid_(.+?)_test_metrics\.npz$', path.name)
    if match is None:
        continue
    run_id = match.group(1)
    event = summary_by_run.loc[run_id]
    data = np.load(path, allow_pickle=True)

    # Saved node arrays already exclude boundary nodes, matching the tester.
    cell_area_all = shp_values(relative_raw_path(event['Cells_Shp_Filepath']), 'area_m2')
    node_types = shp_values(relative_raw_path(event['Nodes_Shp_Filepath']), 'node_type')
    # The dataset/tester treats node_type == 1 as non-boundary.
    cell_area = cell_area_all[node_types == 1]
    pred_volume = np.asarray(data['pred'], dtype=float).squeeze(-1)
    target_volume = np.asarray(data['target'], dtype=float).squeeze(-1)
    if pred_volume.shape[-1] != len(cell_area):
        raise ValueError(f'{path.name}: {pred_volume.shape} does not match {len(cell_area)} non-boundary cell areas')
    pred_depth = pred_volume / cell_area
    target_depth = target_volume / cell_area

    edge_pred = np.asarray(data['edge_pred'], dtype=float).squeeze(-1)
    edge_target = np.asarray(data['edge_target'], dtype=float).squeeze(-1)
    edge_mae = np.mean(np.abs(edge_pred - edge_target))
    edge_length_all = shp_values(
        relative_raw_path(event["Edges_Shp_Filepath"]),
        "fc_length",
    )
    
    edge_types = shp_values(
        relative_raw_path(event["Edges_Shp_Filepath"]),
        "edge_type",
    )
    
    edge_mask = edge_types != 3
    edge_length = edge_length_all[edge_mask]
    
    if edge_pred.shape[-1] != len(edge_length):
        raise ValueError(
            f"{path.name}: {edge_pred.shape} does not match "
            f"{len(edge_length)} edge lengths"
        )
    unit_discharge_mae = np.mean(np.abs(edge_pred / edge_length - edge_target / edge_length))

    rows.append({
        'run_id': run_id,
        'source_file': path.name,
        'depth_mae_m': np.mean(np.abs(pred_depth - target_depth)),
        'discharge_mae_m3_per_s': edge_mae,
        'unit_discharge_mae_m2_per_s': unit_discharge_mae if UNIT_DISCHARGE else np.nan,
        'csi_005m': csi(pred_depth, target_depth, 0.05),
        'csi_030m': csi(pred_depth, target_depth, 0.30),
        'inference_time_s_per_timestep': float(data['inference_time']),
    })

results = pd.DataFrame(rows).sort_values('run_id').reset_index(drop=True)
results.to_csv(OUTPUT_CSV, index=False)
results


,run_id,source_file,depth_mae_m,discharge_mae_m3_per_s,unit_discharge_mae_m2_per_s,csi_005m,csi_030m,inference_time_s_per_timestep
0,100,DUALFloodGNN_2026-09-27_08-17-01_runid_100_tes...,0.111159,0.704153,0.003657,0.715826,0.066985,0.008401
1,81,DUALFloodGNN_2026-09-27_08-17-01_runid_81_test...,0.107831,0.462701,0.002901,0.634349,0.317901,0.020869
2,82,DUALFloodGNN_2026-09-27_08-17-01_runid_82_test...,0.090584,0.408760,0.002152,0.414204,0.754280,0.008569
3,83,DUALFloodGNN_2026-09-27_08-17-01_runid_83_test...,0.139610,0.717825,0.004104,0.455058,0.432363,0.008292
4,84,DUALFloodGNN_2026-09-27_08-17-01_runid_84_test...,0.172073,0.740949,0.004544,0.559543,0.459734,0.008491
5,85,DUALFloodGNN_2026-09-27_08-17-01_runid_85_test...,0.132191,0.724578,0.004476,0.642767,0.550119,0.008510
6,86,DUALFloodGNN_2026-09-27_08-17-01_runid_86_test...,0.072497,0.550535,0.003350,0.659722,0.083352,0.008458
7,87,DUALFloodGNN_2026-09-27_08-17-01_runid_87_test...,0.080926,0.787469,0.004878,0.694527,0.385246,0.008572
8,88,DUALFloodGNN_2026-09-27_08-17-01_runid_88_test...,0.119777,0.288373,0.001609,0.566448,0.025042,0.008397
9,89,DUALFloodGNN_2026-09-27_08-17-01_runid_89_test...,0.086129,1.000247,0.005728,0.732384,0.525038,0.008593


In [17]:
metric_columns = [
    'depth_mae_m', 'discharge_mae_m3_per_s',
    'unit_discharge_mae_m2_per_s', 'csi_005m', 'csi_030m',
    'inference_time_s_per_timestep'
]
summary_stats = pd.DataFrame({
    'mean': results[metric_columns].mean(),
    'sd': results[metric_columns].std(ddof=1).fillna(0.0),
})
print(MODEL_PREFIX)
summary_stats


DUALFloodGNN_2026-09-27_08-17-01


,mean,sd
depth_mae_m,0.112497,0.030180
discharge_mae_m3_per_s,0.670640,0.300746
unit_discharge_mae_m2_per_s,0.003895,0.001787
csi_005m,0.607366,0.100339
csi_030m,0.322630,0.216273
inference_time_s_per_timestep,0.009131,0.002765
